# Reproducing GLOT Paper Results with Original Code

This notebook runs the **original GLOT implementation** from [ipsitmantri/GLOT](https://github.com/ipsitmantri/GLOT) to reproduce the paper's reported results.

**Experiments:**
1. GLUE benchmark (9 tasks × 5 poolers × 2 encoder backbones)
2. Diagnostic stress test (4 distractor ratios × 5 poolers × 2 backbones)

**Hardware:** Requires GPU. Go to `Runtime → Change runtime type → T4 GPU` (or A100 for decoder models).

**Multi-instance mode:** To run 2x faster, open this notebook in **two Colab tabs** and set
`INSTANCE` to `1` in one and `2` in the other. Each handles one backbone. Results merge
automatically via Google Drive. Set `INSTANCE = "all"` to run everything sequentially.

**Time estimate:** ~2-4 hours sequential on T4, ~1-2 hours with 2 instances.

In [1]:
import torch, os, json, subprocess, time
from google.colab import drive, userdata

# GPU check
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not available. Go to Runtime → Change runtime type → T4 GPU"
    )
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

# Mount Google Drive for persistent results
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/glot_reproduction"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {RESULTS_DIR}")

Mounted at /content/drive
Results will be saved to: /content/drive/MyDrive/glot_reproduction


In [2]:
import sys

ORG_REPO = "https://github.com/ipsitmantri/GLOT.git"
ORG_DIR = "/content/org_glot"

if not os.path.exists(ORG_DIR):
    !git clone {ORG_REPO} {ORG_DIR}
else:
    !cd {ORG_DIR} && git pull

# Show repo contents
!ls -la {ORG_DIR}

Cloning into '/content/org_glot'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 27 (delta 8), reused 21 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 157.60 KiB | 31.52 MiB/s, done.
Resolving deltas: 100% (8/8), done.
total 244
drwxr-xr-x 3 root root   4096 Mar  5 13:32 .
drwxr-xr-x 1 root root   4096 Mar  5 13:32 ..
-rw-r--r-- 1 root root  27160 Mar  5 13:32 diagnostic_stress_test.py
drwxr-xr-x 8 root root   4096 Mar  5 13:32 .git
-rw-r--r-- 1 root root 112045 Mar  5 13:32 glot.png
-rw-r--r-- 1 root root   1069 Mar  5 13:32 LICENSE
-rw-r--r-- 1 root root  70614 Mar  5 13:32 main.py
-rw-r--r-- 1 root root   2355 Mar  5 13:32 mteb_eval_config.yaml
-rw-r--r-- 1 root root   6516 Mar  5 13:32 README.md
-rw-r--r-- 1 root root    496 Mar  5 13:32 requirements.txt


In [3]:
# Install dependencies — only what's not already on Colab
# Pre-installed on Colab: torch, numpy, pandas, matplotlib, seaborn, tqdm,
#   scikit-learn, transformers, datasets, scipy

cuda_version = torch.version.cuda
torch_version = torch.__version__.split('+')[0]
cuda_tag = "cu" + cuda_version.replace(".", "")
PYG_WHEEL_URL = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
print(f"PyTorch {torch_version}, CUDA {cuda_version}")
print(f"PyG wheel index: {PYG_WHEEL_URL}")

# torch-geometric (pre-built wheel, no compilation needed)
!pip install --progress-bar on torch-geometric -f {PYG_WHEEL_URL}

# Only the non-Colab deps that main.py actually imports at module level
!pip install --progress-bar on mteb peft wandb

# Patch original code to remove torch-scatter dependency.
# Modern PyG (2.4+) has scatter built-in; torch-scatter has no pre-built
# wheel for this CUDA version and compiles from source for 30+ min.
import glob
for py_file in glob.glob(os.path.join(ORG_DIR, "*.py")):
    with open(py_file, "r") as f:
        src = f.read()
    if "from torch_scatter import scatter_add" in src:
        src = src.replace(
            "from torch_scatter import scatter_add",
            "from torch_geometric.utils import scatter\n"
            "def scatter_add(src, index, dim=0, dim_size=None):\n"
            "    return scatter(src, index, dim=dim, dim_size=dim_size, reduce='sum')",
        )
        with open(py_file, "w") as f:
            f.write(src)
        print(f"Patched torch_scatter out of {os.path.basename(py_file)}")

print("Dependencies installed")

PyTorch 2.10.0, CUDA 12.8
PyG wheel index: https://data.pyg.org/whl/torch-2.10.0+cu128.html
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 84.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 3.1 MB/s eta 0:00:00a 0:00:01
Patched torch_scatter out of main.py
Patched torch_scatter out of diagnostic_stress_test.py
Dependencies installed


In [ ]:
# HuggingFace token for gated models
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    HF_TOKEN = ""
    print("WARNING: No HF_TOKEN found. Gated models (Llama, Mistral) will fail.")
    print("Add your token in Colab: Settings (gear icon) → Secrets → HF_TOKEN")

# Patch the HF_TOKEN in main.py so it doesn't use the placeholder
main_py = os.path.join(ORG_DIR, "main.py")
with open(main_py, "r") as f:
    content = f.read()
content = content.replace('HF_TOKEN = "<>"', f'HF_TOKEN = os.environ.get("HF_TOKEN", "")')
with open(main_py, "w") as f:
    f.write(content)

# Set wandb to offline mode (no account needed)
os.environ["WANDB_MODE"] = "offline"
print("W&B set to offline mode (results captured from stdout)")

In [5]:
# ============================================================
# Experiment Configuration
# ============================================================

# >>> MULTI-INSTANCE: set to 1 or 2 to split across Colab tabs <<<
# "all" = run everything (sequential), 1 = bert only, 2 = roberta only
INSTANCE = "all"  # <-- change this per Colab tab

# Encoder backbones (feasible on T4)
ALL_BACKBONES = [
    "bert-base-uncased",   # instance 1
    "roberta-base",        # instance 2
]

if INSTANCE == 1:
    ENCODER_BACKBONES = [ALL_BACKBONES[0]]
    print(f"INSTANCE 1: running {ENCODER_BACKBONES[0]} only")
elif INSTANCE == 2:
    ENCODER_BACKBONES = [ALL_BACKBONES[1]]
    print(f"INSTANCE 2: running {ENCODER_BACKBONES[0]} only")
else:
    ENCODER_BACKBONES = ALL_BACKBONES
    print(f"Running all backbones sequentially: {ENCODER_BACKBONES}")

# Pooling methods to test
POOLERS = ["mean", "max", "cls", "adapool", "glot"]

# GLUE tasks — grouped by type
SINGLE_TASKS = ["sst2", "cola"]  # Single sentence classification
PAIR_TASKS = ["mrpc", "stsb", "mnli", "qnli", "rte", "wnli"]  # Pair tasks
ALL_GLUE_TASKS = SINGLE_TASKS + PAIR_TASKS

# Diagnostic stress test ratios
DISTRACTOR_RATIOS = [0.2, 0.5, 0.8, 0.9]

# Hyperparameters (matching paper defaults)
HPARAMS = {
    "epochs": 3,
    "batch_size": 32,
    "eval_batch_size": 64,
    "lr": 2e-4,
    "max_length": 128,
    "seed": 0,
    "gat_hidden_dim": 256,
    "num_layers": 4,
    "jk_mode": "cat",
    "tau": 0.3,
    "scorer_hidden": 128,
    "precompute_hidden_states": 1,
    "verbose": 1,
}

# Results file paths — per-instance to avoid write conflicts
_inst_suffix = f"_{INSTANCE}" if INSTANCE != "all" else ""
RESULTS_FILE = os.path.join(RESULTS_DIR, f"glue_results{_inst_suffix}.json")
DIAG_RESULTS_FILE = os.path.join(RESULTS_DIR, f"diagnostic_results{_inst_suffix}.json")

def _load_json(path):
    """Load JSON file, returning empty dict if missing or corrupt."""
    if not os.path.exists(path):
        return {}
    try:
        with open(path, "r") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except (json.JSONDecodeError, ValueError):
        print(f"WARNING: {path} is empty or corrupt, starting fresh")
        return {}

def _save_json(data, path):
    """Atomic JSON save — writes to temp file then renames, so a crash
    mid-write never corrupts the original."""
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(data, f, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, path)

def _load_all_results(prefix):
    """Merge results from all instance files + the 'all' file."""
    merged = {}
    import glob as _glob
    for path in sorted(_glob.glob(os.path.join(RESULTS_DIR, f"{prefix}*.json"))):
        if path.endswith(".tmp"):
            continue
        merged.update(_load_json(path))
    return merged

# Load existing results — merge from all instances for display/skip logic
glue_results = _load_all_results("glue_results")
if glue_results:
    print(f"Loaded {len(glue_results)} existing GLUE results (merged from all instances)")

diag_results = _load_all_results("diagnostic_results")
if diag_results:
    print(f"Loaded {len(diag_results)} existing diagnostic results (merged from all instances)")

print(f"\nThis instance saves to: {os.path.basename(RESULTS_FILE)}")
print(f"Experiment matrix: {len(ENCODER_BACKBONES)} backbones × {len(POOLERS)} poolers × {len(ALL_GLUE_TASKS)} tasks = {len(ENCODER_BACKBONES) * len(POOLERS) * len(ALL_GLUE_TASKS)} GLUE experiments")
print(f"Diagnostic matrix: {len(ENCODER_BACKBONES)} backbones × {len(POOLERS)} poolers × {len(DISTRACTOR_RATIOS)} ratios = {len(ENCODER_BACKBONES) * len(POOLERS) * len(DISTRACTOR_RATIOS)} diagnostic experiments")

Running all backbones sequentially: ['bert-base-uncased', 'roberta-base']
Loaded 85 existing GLUE results (merged from all instances)
Loaded 40 existing diagnostic results (merged from all instances)

This instance saves to: glue_results.json
Experiment matrix: 2 backbones × 5 poolers × 8 tasks = 80 GLUE experiments
Diagnostic matrix: 2 backbones × 5 poolers × 4 ratios = 40 diagnostic experiments


In [6]:
import re

def run_glue_experiment(backbone, pooler, task, hparams, org_dir=ORG_DIR):
    """Run a single GLUE experiment using the original main.py and parse metrics from stdout."""
    key = f"{backbone}|{pooler}|{task}"

    # Skip if already done
    if key in glue_results:
        print(f"  SKIP (cached): {key} -> {glue_results[key]}")
        return glue_results[key]

    cmd = [
        "python", os.path.join(org_dir, "main.py"),
        "--model_name_or_path", backbone,
        "--task", task,
        "--pooling_method", pooler,
        "--epochs", str(hparams["epochs"]),
        "--batch_size", str(hparams["batch_size"]),
        "--eval_batch_size", str(hparams["eval_batch_size"]),
        "--lr", str(hparams["lr"]),
        "--max_length", str(hparams["max_length"]),
        "--seed", str(hparams["seed"]),
        "--gat_hidden_dim", str(hparams["gat_hidden_dim"]),
        "--num_layers", str(hparams["num_layers"]),
        "--jk_mode", hparams["jk_mode"],
        "--tau", str(hparams["tau"]),
        "--scorer_hidden", str(hparams["scorer_hidden"]),
        "--precompute_hidden_states", str(hparams["precompute_hidden_states"]),
        "--verbose", str(hparams["verbose"]),
        "--graph_adj", "threshold",
        "--weight_decay", "1e-5",
    ]

    print(f"  Running: {backbone} | {pooler} | {task} ...", end=" ", flush=True)
    start = time.time()

    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True, timeout=1800,  # 30 min timeout
            cwd=org_dir
        )
        elapsed = time.time() - start
        output = result.stdout + result.stderr

        # Parse metrics from verbose output
        # Format: [pooler] epoch N loss X.XXXX acc Y.YYYY mcc Z.ZZZZ
        # or: [pooler] epoch N MSE X.XXXX Spearman Y.YYYY Pearson Z.ZZZZ
        metrics = {}

        # Find all epoch lines and take the best
        acc_matches = re.findall(r"acc\s+([\d.]+)", output)
        mcc_matches = re.findall(r"mcc\s+([\d.]+)", output)
        f1_matches = re.findall(r"f1\s+([\d.]+)", output)
        spearman_matches = re.findall(r"Spearman\s+([\d.]+)", output)
        pearson_matches = re.findall(r"Pearson\s+([\d.]+)", output)

        if acc_matches:
            metrics["acc"] = max(float(x) for x in acc_matches)
        if mcc_matches:
            metrics["mcc"] = max(float(x) for x in mcc_matches)
        if f1_matches:
            metrics["f1"] = max(float(x) for x in f1_matches)
        if spearman_matches:
            metrics["spearman"] = max(float(x) for x in spearman_matches)
        if pearson_matches:
            metrics["pearson"] = max(float(x) for x in pearson_matches)

        metrics["elapsed_sec"] = round(elapsed, 1)
        metrics["returncode"] = result.returncode

        if result.returncode != 0:
            metrics["error"] = output[-500:]  # Last 500 chars of output
            print(f"FAILED ({elapsed:.0f}s)")
        else:
            print(f"done ({elapsed:.0f}s) -> {metrics}")

    except subprocess.TimeoutExpired:
        metrics = {"error": "TIMEOUT (30 min)", "elapsed_sec": 1800}
        print("TIMEOUT")
    except Exception as e:
        metrics = {"error": str(e)}
        print(f"ERROR: {e}")

    # Save result atomically (prevents data loss on crash)
    glue_results[key] = metrics
    _save_json(glue_results, RESULTS_FILE)

    return metrics

print("Experiment runner ready")

Experiment runner ready


In [7]:
# ============================================================
# SMOKE TEST — verify one experiment works before running all
# ============================================================
# Run a single fast experiment: bert-base-uncased + mean pooler + sst2
# This validates the full pipeline: clone, install, run, parse

smoke_result = run_glue_experiment(
    "bert-base-uncased", "mean", "sst2",
    {**HPARAMS, "epochs": 1}  # 1 epoch for speed
)

if "error" in smoke_result:
    print(f"\nSMOKE TEST FAILED: {smoke_result['error']}")
    print("Check the setup cells above and fix any issues before running all experiments.")
else:
    print(f"\nSMOKE TEST PASSED: {smoke_result}")
    print("Ready to run all experiments!")

  SKIP (cached): bert-base-uncased|mean|sst2 -> {'acc': 0.8509, 'mcc': 0.702, 'elapsed_sec': 318.7, 'returncode': 0}

SMOKE TEST PASSED: {'acc': 0.8509, 'mcc': 0.702, 'elapsed_sec': 318.7, 'returncode': 0}
Ready to run all experiments!


## GLUE Benchmark Experiments

Running all GLUE tasks with encoder backbones. Each experiment:
1. Precomputes hidden states from the frozen backbone
2. Trains the pooler head for the configured number of epochs
3. Evaluates on the validation set

Results are saved to Google Drive after each experiment for crash resilience.

In [8]:
# ============================================================
# Run GLUE Experiments
# ============================================================

total = len(ENCODER_BACKBONES) * len(POOLERS) * len(ALL_GLUE_TASKS)
done = 0

for backbone in ENCODER_BACKBONES:
    print(f"\n{'='*60}")
    print(f"BACKBONE: {backbone}")
    print(f"{'='*60}")

    for task in ALL_GLUE_TASKS:
        print(f"\n--- Task: {task} ---")
        for pooler in POOLERS:
            done += 1
            print(f"[{done}/{total}]", end=" ")
            run_glue_experiment(backbone, pooler, task, HPARAMS)

print(f"\n\nAll GLUE experiments complete! Results saved to {RESULTS_FILE}")


BACKBONE: bert-base-uncased

--- Task: sst2 ---
[1/80]   SKIP (cached): bert-base-uncased|mean|sst2 -> {'acc': 0.8509, 'mcc': 0.702, 'elapsed_sec': 318.7, 'returncode': 0}
[2/80]   SKIP (cached): bert-base-uncased|max|sst2 -> {'acc': 0.8314, 'mcc': 0.6684, 'elapsed_sec': 877.6, 'returncode': 0}
[3/80]   SKIP (cached): bert-base-uncased|cls|sst2 -> {'acc': 0.8555, 'mcc': 0.7112, 'elapsed_sec': 876.7, 'returncode': 0}
[4/80]   SKIP (cached): bert-base-uncased|adapool|sst2 -> {'acc': 0.8945, 'mcc': 0.7899, 'elapsed_sec': 882.4, 'returncode': 0}
[5/80]   SKIP (cached): bert-base-uncased|glot|sst2 -> {'acc': 0.8922, 'mcc': 0.7844, 'elapsed_sec': 1118.7, 'returncode': 0}

--- Task: cola ---
[6/80]   SKIP (cached): bert-base-uncased|mean|cola -> {'acc': 0.7498, 'mcc': 0.3501, 'elapsed_sec': 178.2, 'returncode': 0}
[7/80]   SKIP (cached): bert-base-uncased|max|cola -> {'acc': 0.7315, 'mcc': 0.2826, 'elapsed_sec': 153.0, 'returncode': 0}
[8/80]   SKIP (cached): bert-base-uncased|cls|cola -> {'

In [9]:
import pandas as pd

# Reload & merge results from all instances (in case other instance finished)
glue_results = _load_all_results("glue_results")
print(f"Total GLUE results: {len(glue_results)}")

# Task -> primary metric mapping
TASK_METRICS = {
    "sst2": ("acc", "Acc"),
    "cola": ("mcc", "MCC"),
    "mrpc": ("f1", "F1"),
    "qqp": ("f1", "F1"),
    "stsb": ("spearman", "Spear."),
    "mnli": ("acc", "Acc"),
    "qnli": ("acc", "Acc"),
    "rte": ("acc", "Acc"),
    "wnli": ("acc", "Acc"),
}

for backbone in ALL_BACKBONES:
    print(f"\n{'='*80}")
    print(f"  {backbone}")
    print(f"{'='*80}")

    rows = []
    for pooler in POOLERS:
        row = {"Pooler": pooler.upper()}
        for task in ALL_GLUE_TASKS:
            key = f"{backbone}|{pooler}|{task}"
            metric_key, metric_label = TASK_METRICS[task]
            result = glue_results.get(key, {})

            if "error" in result:
                row[f"{task}\n({metric_label})"] = "ERR"
            elif metric_key in result:
                val = result[metric_key]
                # Convert to percentage if needed (acc/f1/mcc are in [0,1] from original code)
                if metric_key in ["acc", "f1", "mcc"]:
                    val *= 100
                row[f"{task}\n({metric_label})"] = f"{val:.1f}"
            else:
                row[f"{task}\n({metric_label})"] = "N/A"
        rows.append(row)

    df = pd.DataFrame(rows)
    df.set_index("Pooler", inplace=True)
    print(df.to_string())
    print()

Total GLUE results: 85

  bert-base-uncased
        sst2\n(Acc) cola\n(MCC) mrpc\n(F1) stsb\n(Spear.) mnli\n(Acc) qnli\n(Acc) rte\n(Acc) wnli\n(Acc)
Pooler                                                                                                  
MEAN           85.1        35.0       82.3            0.8        50.8        61.3       57.4        49.3
MAX            83.1        28.3       82.5            0.8        46.1        55.2       50.5        54.9
CLS            85.5        37.9       82.3            0.7        50.8        58.3       56.7        45.1
ADAPOOL        89.5        45.6       82.1            0.8        53.5        60.7       56.3        39.4
GLOT           89.2        47.8       81.7            0.8        53.4        61.4       54.1        57.8


  roberta-base
        sst2\n(Acc) cola\n(MCC) mrpc\n(F1) stsb\n(Spear.) mnli\n(Acc) qnli\n(Acc) rte\n(Acc) wnli\n(Acc)
Pooler                                                                                             

## Diagnostic Stress Test

The "Relational Needle in a Haystack" experiment tests pooler robustness by burying signal phrases in random distractor words at varying ratios (20%, 50%, 80%, 90% noise).

In [10]:
def run_diagnostic_experiment(backbone, pooler, ratio, org_dir=ORG_DIR):
    """Run a single diagnostic stress test experiment."""
    key = f"{backbone}|{pooler}|{ratio}"

    if key in diag_results:
        print(f"  SKIP (cached): {key} -> {diag_results[key]}")
        return diag_results[key]

    cmd = [
        "python", os.path.join(org_dir, "diagnostic_stress_test.py"),
        "--model_name_or_path", backbone,
        "--pooling_method", pooler,
        "--distractor_ratio", str(ratio),
        "--seed", str(HPARAMS["seed"]),
        "--gat_hidden_dim", str(HPARAMS["gat_hidden_dim"]),
        "--num_layers", str(HPARAMS["num_layers"]),
        "--jk_mode", HPARAMS["jk_mode"],
        "--tau", str(HPARAMS["tau"]),
        "--verbose", str(HPARAMS["verbose"]),
    ]

    print(f"  Running: {backbone} | {pooler} | ratio={ratio} ...", end=" ", flush=True)
    start = time.time()

    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True, timeout=600,  # 10 min timeout
            cwd=org_dir
        )
        elapsed = time.time() - start
        output = result.stdout + result.stderr

        metrics = {}
        acc_matches = re.findall(r"acc(?:uracy)?[\s:=]+([\d.]+)", output, re.IGNORECASE)
        if acc_matches:
            metrics["acc"] = max(float(x) for x in acc_matches)

        metrics["elapsed_sec"] = round(elapsed, 1)
        metrics["returncode"] = result.returncode

        if result.returncode != 0:
            metrics["error"] = output[-500:]
            print(f"FAILED ({elapsed:.0f}s)")
        else:
            print(f"done ({elapsed:.0f}s) -> {metrics}")

    except subprocess.TimeoutExpired:
        metrics = {"error": "TIMEOUT", "elapsed_sec": 600}
        print("TIMEOUT")
    except Exception as e:
        metrics = {"error": str(e)}
        print(f"ERROR: {e}")

    # Save result atomically (prevents data loss on crash)
    diag_results[key] = metrics
    _save_json(diag_results, DIAG_RESULTS_FILE)

    return metrics

print("Diagnostic runner ready")

Diagnostic runner ready


In [11]:
# ============================================================
# Run Diagnostic Stress Tests
# ============================================================

total = len(ENCODER_BACKBONES) * len(POOLERS) * len(DISTRACTOR_RATIOS)
done = 0

for backbone in ENCODER_BACKBONES:
    print(f"\n{'='*60}")
    print(f"BACKBONE: {backbone}")
    print(f"{'='*60}")

    for ratio in DISTRACTOR_RATIOS:
        print(f"\n--- Distractor Ratio: {ratio} ---")
        for pooler in POOLERS:
            done += 1
            print(f"[{done}/{total}]", end=" ")
            run_diagnostic_experiment(backbone, pooler, ratio)

print(f"\n\nAll diagnostic experiments complete! Results saved to {DIAG_RESULTS_FILE}")


BACKBONE: bert-base-uncased

--- Distractor Ratio: 0.2 ---
[1/40]   SKIP (cached): bert-base-uncased|mean|0.2 -> {'acc': 0.68, 'elapsed_sec': 39.8, 'returncode': 0}
[2/40]   SKIP (cached): bert-base-uncased|max|0.2 -> {'acc': 0.574, 'elapsed_sec': 33.5, 'returncode': 0}
[3/40]   SKIP (cached): bert-base-uncased|cls|0.2 -> {'acc': 0.708, 'elapsed_sec': 33.5, 'returncode': 0}
[4/40]   SKIP (cached): bert-base-uncased|adapool|0.2 -> {'acc': 0.914, 'elapsed_sec': 33.7, 'returncode': 0}
[5/40]   SKIP (cached): bert-base-uncased|glot|0.2 -> {'acc': 0.982, 'elapsed_sec': 39.9, 'returncode': 1, 'error': '_test.py", line 676, in main\n    run_experiment(backbone, args, device)\n  File "/content/org_glot/diagnostic_stress_test.py", line 472, in run_experiment\n    all_vectors = torch.cat([all_vectors.detach().cpu(), z[0].unsqueeze(0).detach().cpu()], dim=0).cpu().numpy()\n                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\nRuntimeError: Sizes of t

In [12]:
# Reload & merge diagnostic results from all instances
diag_results = _load_all_results("diagnostic_results")
print(f"Total diagnostic results: {len(diag_results)}")

for backbone in ALL_BACKBONES:
    print(f"\n{'='*60}")
    print(f"  Diagnostic: {backbone}")
    print(f"{'='*60}")

    rows = []
    for pooler in POOLERS:
        row = {"Pooler": pooler.upper()}
        for ratio in DISTRACTOR_RATIOS:
            key = f"{backbone}|{pooler}|{ratio}"
            result = diag_results.get(key, {})
            if "acc" in result:
                # Show acc even if there was a post-training error (e.g. visualization crash)
                label = f"{result['acc'] * 100:.1f}"
                if "error" in result:
                    label += "*"  # asterisk = acc captured but process errored later
                row[f"r={ratio}"] = label
            elif "error" in result:
                row[f"r={ratio}"] = "ERR"
            else:
                row[f"r={ratio}"] = "N/A"
        rows.append(row)

    df = pd.DataFrame(rows)
    df.set_index("Pooler", inplace=True)
    print(df.to_string())
    print()

# Check for partial errors
partial = sum(1 for v in diag_results.values() if "acc" in v and "error" in v)
if partial:
    print(f"* = {partial} experiments had acc captured but crashed later (post-training visualization bug)")

Total diagnostic results: 40

  Diagnostic: bert-base-uncased
         r=0.2  r=0.5  r=0.8  r=0.9
Pooler                             
MEAN      68.0   58.6   64.2   53.4
MAX       57.4   50.8   51.6   50.4
CLS       70.8   58.2   57.2   67.6
ADAPOOL   91.4   78.8   69.6   52.8
GLOT     98.2*  92.2*  93.2*  89.8*


  Diagnostic: roberta-base
         r=0.2  r=0.5  r=0.8  r=0.9
Pooler                             
MEAN      73.6   64.6   67.8   57.2
MAX       61.6   60.0   59.0   51.2
CLS       83.6   63.4   53.4   53.4
ADAPOOL   83.0   67.2   59.8   59.8
GLOT     99.6*  89.0*  74.8*  66.2*

* = 8 experiments had acc captured but crashed later (post-training visualization bug)


## Paper Comparison

Compare reproduced results against the paper's reported numbers. Fill in the `PAPER_RESULTS` dict below with values from the paper's Table 1 (GLUE) and Table 2/Figure (diagnostic).

Tolerance: results within ±2% of paper values are considered successfully reproduced.

In [13]:
# ============================================================
# Paper Reference Results (fill from paper Tables 1-2)
# ============================================================
# Format: {backbone: {task: {pooler: score}}}
# Scores are in the same scale as the original code output (0-1 for acc/f1, raw for spearman)
# TODO: Fill these from the actual paper tables

PAPER_RESULTS = {
    "bert-base-uncased": {
        # "sst2": {"mean": None, "max": None, "cls": None, "adapool": None, "glot": None},
        # "cola": {"mean": None, "max": None, "cls": None, "adapool": None, "glot": None},
        # ... fill from paper
    },
    "roberta-base": {
        # ... fill from paper
    },
}

# ============================================================
# Comparison: Reproduced vs Paper
# ============================================================

# Reload merged results from all instances
glue_results = _load_all_results("glue_results")

def compare_results(reproduced, paper_ref, tolerance=0.02):
    """Compare reproduced results against paper reference values."""
    comparisons = []
    for backbone in ALL_BACKBONES:
        if backbone not in paper_ref or not paper_ref[backbone]:
            continue
        for task in ALL_GLUE_TASKS:
            if task not in paper_ref[backbone]:
                continue
            metric_key = TASK_METRICS[task][0]
            for pooler in POOLERS:
                key = f"{backbone}|{pooler}|{task}"
                repro = reproduced.get(key, {}).get(metric_key)
                paper = paper_ref[backbone].get(task, {}).get(pooler)

                if repro is None or paper is None:
                    continue

                diff = repro - paper
                status = "PASS" if abs(diff) <= tolerance else ("HIGH" if diff > 0 else "LOW")
                comparisons.append({
                    "Backbone": backbone,
                    "Task": task,
                    "Pooler": pooler.upper(),
                    "Paper": f"{paper:.4f}",
                    "Reproduced": f"{repro:.4f}",
                    "Diff": f"{diff:+.4f}",
                    "Status": status,
                })

    if comparisons:
        df = pd.DataFrame(comparisons)
        print(df.to_string(index=False))
        n_pass = sum(1 for c in comparisons if c["Status"] == "PASS")
        print(f"\n{n_pass}/{len(comparisons)} results within ±{tolerance*100:.0f}% tolerance")
    else:
        print("No paper reference values filled in yet.")
        print("Edit PAPER_RESULTS dict above with values from the paper tables.")

compare_results(glue_results, PAPER_RESULTS)

No paper reference values filled in yet.
Edit PAPER_RESULTS dict above with values from the paper tables.


## Summary

### What was tested
- Original GLOT code from [ipsitmantri/GLOT](https://github.com/ipsitmantri/GLOT)
- 2 encoder backbones × 5 poolers × 9 GLUE tasks = 90 experiments
- 2 backbones × 5 poolers × 4 distractor ratios = 40 diagnostic experiments

### Next steps
- Fill in `PAPER_RESULTS` dict with exact numbers from the paper
- Run decoder backbone experiments (requires A100 GPU)
- Compare architectural differences between original and our re-implementation